In [ ]:
# pip install h5py

In [ ]:
#Library imports
import h5py
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
relative_path = '../data/new_Input_CP_Studies_llqq_LinearTerm_29_September2025.h5'
with h5py.File(relative_path) as f:
    df = pd.DataFrame(f['LargeRJet']['1d'][:])
    df_2d = f['LargeRJet']['2d'][:]

In [ ]:
print(df_2d)

In [ ]:
df.head()

In [ ]:
df.keys()

# Data Key Index:

## Leading index:

FJ: Fat Jet

LeadingSubJet: Smaller jet found within Fat Jet

NegLep, PosLep: Negative and Positively charged leptons

Vlep: reconstructed vector bosons (Z) that decay leptonically

## Secondary Index
E: Energy

Eta: Pseudorapidity

Phi: Azimuthal Angle

pT: transverse momentum

flavour: flavour of quark that initiated jet

mass: invariant mass 

## Extra
Leg_pT_balance: lepton transverse momentum balance

Lumi_weight: luminosity (used for determining signal vs background)

phi: Unsure

phi1: unsure

costhetastar: cosine of decay product in decaying particle frame

costheta1: unsure

costheta2: unsure

In [ ]:
df.Lumi_weight

In [ ]:
X = df.drop(columns=['Lumi_weight'])
y = df['Lumi_weight']

In [ ]:
y

In [ ]:
df.describe()

In [ ]:
zero_mi_features = [
 'FJ_eta', 'LeadingSubJet_pT', 'LeadingSubJet_Phi', 'NegLep_pT', 'Lep_pT_balance',
 'NegLep_Eta', 'NegLep_Phi', 'FJ_mass', 'LeadingSubJet_Eta', 'SubLeadingSubJet_E',
 'PosLep_Eta', 'SubLeadingSubJet_Eta', 'PosLep_Phi', 'PosLep_pT', 'Vlep_E',
 'Vlep_mass', 'cosThetaStar', 'costheta1', 'costheta2'
]

In [ ]:
data_zero = df[zero_mi_features]
data_zero.std()

In [ ]:
scaled_data = (X - X.mean()) / X.std()

In [ ]:
f = plt.figure(figsize=(19, 15))
plt.matshow(scaled_data.corr(), fignum=f.number)
plt.xticks(range(scaled_data.select_dtypes(['number']).shape[1]), scaled_data.select_dtypes(['number']).columns, fontsize=14, rotation=45)
plt.yticks(range(scaled_data.select_dtypes(['number']).shape[1]), scaled_data.select_dtypes(['number']).columns, fontsize=14)
cb = plt.colorbar()
cb.ax.tick_params(labelsize=14)
plt.title('Correlation Matrix', fontsize=16);

In [ ]:
y[y>0] = 1
y[y<0] = 0

In [ ]:
print("Number of signal events:", sum(y==1))
print("Number of background events:", sum(y==0))

In [ ]:
y = np.array(y)

# New dataset exploration


In [ ]:
relative_path = '../data/new_Input_CP_Studies_llqq_LinearTerm_13th_October2025.h5'
with h5py.File(relative_path) as f:
    df = pd.DataFrame(f['LargeRJet']['1d'][:])
    df_gen = pd.DataFrame(f['LargeRJet'])
    df_2d = f['LargeRJet']['2d'][:]

In [ ]:
df.shape

In [ ]:
df_2d.shape

In [ ]:
df_2d.dtype.names

In [ ]:
df_2d['constituent_D0'].ravel()

In [ ]:
df_2d

In [ ]:
df_2d.dtype.names

In [ ]:
n_samples, n_constituents = df_2d.shape
flat_data = {name: df_2d[name].ravel() for name in df_2d.dtype.names}

df_flat = pd.DataFrame(flat_data)

index = pd.MultiIndex.from_product([range(n_samples), range(n_constituents)], names=['sample', 'constituent'])
df_flat.index = index

print(df_flat.head())

In [ ]:
df_flat.shape

In [ ]:
df_gen.head()

In [ ]:
df.columns

In [ ]:
labels = df['Lumi_weight'].values
labels[labels > 0] = 1
labels[labels < 0] = 0

In [ ]:

def structured_to_unstructured(arr):
    #Shape: (n_events, max_constituents, n_features) -> (218023, 60, 8), Convert to 3d array
    new_arr = np.zeros(arr.shape + (len(arr.dtype.names),), dtype=np.float32)
    for i, name in enumerate(arr.dtype.names):
        new_arr[..., i] = arr[name]
    return new_arr


In [ ]:
constituent_features = structured_to_unstructured(df_2d)

In [ ]:
constituent_features.shape

In [ ]:
valid_constituent_mask = constituent_features[:, :, -1] > 0
constituent_features = np.nan_to_num(constituent_features, nan=0.0)

In [ ]:
print(f"Constituent feature shape: {constituent_features.shape}")
print(f"Number of events: {len(labels)}")

In [ ]:
constituent_features

In [ ]:
path = '../graphdata/gnn_discriminant_scores_validation.npz'
data = np.load(path)
discriminant_scores = data['discriminant_scores']
y_true = data['y_true']
y_pred = data['y_pred']
lumi_weights = data['lumi_weights']

In [ ]:
trues = np.sum(y_true)
print('number of true positive labels: ' +str(trues) )
print('number of true negative labels: ' +str(len(y_true)-trues) )

In [ ]:
trues = np.sum(y_pred)
print('number of predicted positive labels: ' +str(trues) )
print('number of predicted negative labels: ' +str(len(y_pred)-trues) )

In [ ]:
print(np.unique(discriminant_scores, return_counts=True)) #Predicting same disciminants for all GNN? is it only considering 1 event??

In [ ]:
lumi_weights 

In [ ]:
lumi_bools = lumi_weights > 0

In [ ]:
ylist = list(y_true + lumi_bools)
print(ylist.count(0)) #should be 27262
print(ylist.count(1)) #should be 0
print(ylist.count(2)) #should be 27244

Labels and Lumi weights are corresponding to each other correctly in processed data

In [ ]:
relative_path = '../data/s2286706/new_Input_CP_Studies_llqq_LinearTerm_20th_October2025.h5'
with h5py.File(relative_path) as f:
    df = pd.DataFrame(f['LargeRJet']['1d'][:])

In [ ]:
df.columns

In [ ]:
train = df[df['EventNumber'] % 2 == 0]
train.columns

In [ ]:
train.shape

In [ ]:
relative_path = '../data/new_Input_CP_Studies_llqq_LinearTerm_13th_October2025.h5'
with h5py.File(relative_path) as f:
    df = pd.DataFrame(f['LargeRJet']['1d'][:])

df.columns

In [ ]:
df.shape

In [ ]:
import h5py
import pandas as pd

In [ ]:
"C:\Users\colel\OneDrive\Documents\UoE_UG\Y4\SH_Project\SH-Project\data\new_Input_CP_Studies_llqq_LinearTerm_13th_October2025.h5"

In [ ]:
relative_path = '../data/new_Input_CP_Studies_llqq_LinearTerm_13th_October2025.h5'
with h5py.File(relative_path) as f:
    data = f['LargeRJet']['2d'][:]

raveled = pd.DataFrame(data['constituent_D0'].ravel())
dropped = raveled.dropna()
print(dropped.shape)
print(raveled.shape)
print(data.shape)
print(data[0])

In [ ]:
print(data.shape)
print(data[0].shape)   
print(data[0][0])

In [ ]:
import numpy as np

In [ ]:
num_events = data.shape[0]
num_nodes_per_event = data.shape[1]
num_features = len(data.dtype.names)

In [ ]:
reshaped_data = data.view(np.float32).reshape(data.shape[0], data.shape[1], num_features)
print(f"Original data dtype: {data.dtype}")
print(f"New data shape: {reshaped_data.shape}")
print(f"New data dtype: {reshaped_data.dtype}")

In [ ]:
feature_names = list(data.dtype.names)
print(feature_names)

In [ ]:
import torch

In [ ]:
graph_list = []
for i in range(reshaped_data.shape[0]):
    node_features = torch.tensor(reshaped_data[i],dtype = torch.float)
    eta_idx = feature_names.index('constituent_eta')
    phi_idx = feature_names.index('constituent_phi')
    pos = node_features[:, [eta_idx, phi_idx]]
    edge_index = knn_graph(pos, k=6, loop=False)

    graph_label = torch.tensor([labels[i]], dtype=torch.long)
    
    graph_data = Data(
        x=node_features, 
        edge_index=edge_index, 
        y=graph_label,
        feature_names=feature_names
    )

    graph_list.append(graph_data)

In [ ]:
reshaped_data.columns

In [ ]:
relative_path = "../data/s2286706/new_Input_CP_Studies_llqq_LinearTerm_20th_October2025.h5"
with h5py.File(relative_path) as f:
    data = pd.DataFrame(f['LargeRJet']['2d'])
data.head()